In [ ]:
# ============================================================
# RAG CONFIGURATION REGENERATION
# ============================================================
#
# PURPOSE
# -------
# Regenerate the SAME 9 RAG configuration summaries using
# one final, explicitly documented summary-generation model.
#
# This notebook DOES NOT:
# - change the selected patient
# - change preprocessing
# - change chunking definitions
# - change embedding models
# - change the retrieval query
# - change Top-K
# - change chronological reordering
#
# It only regenerates the 9 summary outputs under one final
# OpenAI generation model so downstream evaluations are
# internally consistent and reproducible.
#
# FINAL SUMMARY GENERATOR
# -----------------------
# Provider: OpenAI
# Model: gpt-5.6-luna
# Temperature: model default
#
# OUTPUT
# ------
# rag_configuration_summaries_final.json
# ============================================================

import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from google import genai
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoModel, AutoTokenizer
import torch


In [7]:
# ============================================================
# FROZEN EXPERIMENT SETTINGS
# ============================================================

SUMMARY_PROVIDER = "OpenAI"
SUMMARY_MODEL = "gpt-5.6-luna"

GEMINI_EMBEDDING_MODEL = "gemini-embedding-001"
BGE_EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"

TOP_K = 20

SELECTED_PERSON_ID = "c6c45c39-cd73-49dd-818d-0a7865fe8a7f"

RAG_QUERY = """
Retrieve the clinically relevant information needed to produce a
comprehensive longitudinal summary of this patient's clinical history,
including major diagnoses, treatments, investigations, clinical
progression, and outcomes.
""".strip()

CHUNKING_STRATEGIES = [
    "Whole",
    "Fixed",
    "Section",
]

EMBEDDING_STRATEGIES = [
    "Gemini",
    "BGE",
    "MedCPT",
]

print("==========================================")
print("RAG CONFIGURATION REGENERATION")
print("==========================================")
print("Summary provider :", SUMMARY_PROVIDER)
print("Summary model    :", SUMMARY_MODEL)
print("Selected patient :", SELECTED_PERSON_ID)
print("Top-K            :", TOP_K)
print("Chunking         :", CHUNKING_STRATEGIES)
print("Embeddings       :", EMBEDDING_STRATEGIES)
print("==========================================")

RAG CONFIGURATION REGENERATION
Summary provider : OpenAI
Summary model    : gpt-5.6-luna
Selected patient : c6c45c39-cd73-49dd-818d-0a7865fe8a7f
Top-K            : 20
Chunking         : ['Whole', 'Fixed', 'Section']
Embeddings       : ['Gemini', 'BGE', 'MedCPT']


In [8]:
# ============================================================
# FINAL SUMMARY-GENERATION CLIENT
#
# This client goes DIRECTLY to OpenAI.
# It does NOT use Groq.
# ============================================================

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

print("Client class :", type(openai_client).__name__)
print("Client module:", type(openai_client).__module__)
print("Provider     :", SUMMARY_PROVIDER)
print("Model        :", SUMMARY_MODEL)

Client class : OpenAI
Client module: openai
Provider     : OpenAI
Model        : gpt-5.6-luna


In [9]:
# ============================================================
# SMALL CONNECTION TEST
#
# This confirms:
# 1. OPENAI_API_KEY works.
# 2. The frozen model is accessible.
# 3. We are not accidentally calling Groq.
# ============================================================

test_response = openai_client.chat.completions.create(
    model=SUMMARY_MODEL,
    messages=[
        {
            "role": "user",
            "content": "Return exactly: OpenAI summary model ready"
        }
    ]
)

print(test_response.choices[0].message.content)

OpenAI summary model ready


In [10]:
# ============================================================
# LOAD + PREPROCESS SOURCE NOTES
#
# Same preprocessing used in the original RAG experiment.
# ============================================================

notes = pd.read_csv("../data/raw/clinical_notes.csv")

notes_clean = notes[
    notes["clean_note_text"].astype(str).str.strip() != "#NAME?"
].copy()

notes_dedup = (
    notes_clean
    .sort_values(["person_id", "creation_timestamp"])
    .drop_duplicates(
        subset=["person_id", "clean_note_text"],
        keep="first"
    )
    .reset_index(drop=True)
)

SELECTED_PERSON_ID = "c6c45c39-cd73-49dd-818d-0a7865fe8a7f"

patient_notes = (
    notes_dedup[
        notes_dedup["person_id"] == SELECTED_PERSON_ID
    ]
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

print("Original notes:", len(notes))
print("After cleaning:", len(notes_clean))
print("After deduplication:", len(notes_dedup))
print("Selected patient notes:", len(patient_notes))

Original notes: 1602
After cleaning: 1595
After deduplication: 1103
Selected patient notes: 45


In [11]:
# ============================================================
# BUILD THE THREE FROZEN CHUNKING STRATEGIES
# ============================================================
#
# 1. WHOLE NOTE
#    One clinical note = one chunk
#
# 2. FIXED-SIZE
#    150 words with 30-word overlap
#
# 3. SECTION-AWARE
#    Split notes by section structure / headings
#
# These definitions should match the original 3x3 RAG experiment.
# ============================================================


# -------------------------
# 1. WHOLE-NOTE CHUNKS
# -------------------------

whole_chunks = (
    patient_notes[
        [
            "person_id",
            "creation_timestamp",
            "clean_note_text"
        ]
    ]
    .copy()
    .rename(columns={
        "clean_note_text": "chunk_text"
    })
)

whole_chunks["chunk_id"] = range(len(whole_chunks))

print("Whole-note chunks:", len(whole_chunks))

Whole-note chunks: 45


In [12]:
# -------------------------
# 2. FIXED-SIZE CHUNKS
# -------------------------

FIXED_CHUNK_SIZE = 150
FIXED_CHUNK_OVERLAP = 30


def chunk_text_fixed(text, chunk_size=150, overlap=30):
    words = str(text).split()

    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size

        chunk = " ".join(
            words[start:end]
        )

        if chunk.strip():
            chunks.append(chunk)

        if end >= len(words):
            break

        start += chunk_size - overlap

    return chunks

In [13]:
fixed_records = []

for note_index, row in patient_notes.iterrows():

    note_chunks = chunk_text_fixed(
        row["clean_note_text"],
        chunk_size=FIXED_CHUNK_SIZE,
        overlap=FIXED_CHUNK_OVERLAP
    )

    for chunk_index, chunk_text in enumerate(note_chunks):

        fixed_records.append({
            "person_id": row["person_id"],
            "creation_timestamp": row["creation_timestamp"],
            "source_note_index": int(note_index),
            "within_note_chunk_index": chunk_index,
            "chunk_text": chunk_text
        })

fixed_chunks = pd.DataFrame(
    fixed_records
)

fixed_chunks["chunk_id"] = range(
    len(fixed_chunks)
)

print("Fixed-size chunks:", len(fixed_chunks))

Fixed-size chunks: 56


In [14]:
# ============================================================
# CHUNKING STRATEGY 3 — SECTION-AWARE
#
# Uses a fixed list of recognized clinical section headings.
#
# Rules:
# - Keep heading + section content together.
# - Remove heading-only empty sections.
# - Preserve any text before the first recognized heading.
# - If fewer than two usable section boundaries are detected,
#   keep the complete note as one chunk.
#
# IMPORTANT:
# This is the SAME section-aware chunking logic used in the
# original 3 x 3 RAG configuration experiment.
# ============================================================

SECTION_HEADINGS = [
    "Presenting Complaint",
    "History of Presenting Illness",
    "History of Present Illness",
    "HPI",
    "Review of Systems",
    "Past Medical History",
    "PMH",
    "Medications",
    "Medication",
    "Allergies",
    "Social History",
    "Family History",
    "On Examination",
    "Examination",
    "Observations",
    "Investigations",
    "Test Results",
    "Results",
    "Assessment",
    "Impression",
    "Diagnosis",
    "Treatment",
    "Plan",
]

SECTION_PATTERN = re.compile(
    rf"(?im)^(?:{'|'.join(map(re.escape, SECTION_HEADINGS))})\s*:?\s*$"
)


def is_heading_only(text: str) -> bool:
    """
    Return True when a chunk contains only a recognized
    section heading and no clinical content.
    """
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    if len(lines) != 1:
        return False

    return bool(
        SECTION_PATTERN.fullmatch(lines[0])
    )


def split_by_sections(text: str) -> list[str]:
    """
    Split one clinical note using the frozen section headings.

    If usable section boundaries are not detected,
    return the complete note unchanged.
    """
    matches = list(
        SECTION_PATTERN.finditer(text)
    )

    if len(matches) < 2:
        return [text]

    chunks = []

    prefix = text[:matches[0].start()].strip()

    if prefix:
        chunks.append(prefix)

    for index, match in enumerate(matches):

        start = match.start()

        if index + 1 < len(matches):
            end = matches[index + 1].start()
        else:
            end = len(text)

        section = text[start:end].strip()

        if section and not is_heading_only(section):
            chunks.append(section)

    return chunks


def create_section_chunks(
    notes_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Create section-aware chunks from clinical notes.

    Notes without detectable section structure remain whole.
    """
    records = []

    for row in notes_df.itertuples(index=False):

        text_chunks = split_by_sections(
            row.clean_note_text
        )

        for chunk_index, chunk_text in enumerate(text_chunks):

            records.append(
                {
                    "person_id": row.person_id,
                    "admission_id": row.admission_id,
                    "clinical_note_id": row.clinical_note_id,
                    "creation_timestamp": row.creation_timestamp,
                    "note_subject": row.note_subject,
                    "note_type": row.note_type,
                    "chunk_index": chunk_index,
                    "chunk_id": (
                        f"{row.clinical_note_id}"
                        f"_section_{chunk_index}"
                    ),
                    "chunk_strategy": "section",
                    "chunk_text": chunk_text,
                }
            )

    return pd.DataFrame(records)


section_chunks = create_section_chunks(
    patient_notes
)

print("Section-aware chunks:", len(section_chunks))

Section-aware chunks: 148


In [15]:
chunk_summary = pd.DataFrame(
    {
        "strategy": [
            "Whole note",
            "Fixed words",
            "Section aware",
        ],
        "num_chunks": [
            len(whole_chunks),
            len(fixed_chunks),
            len(section_chunks),
        ],
        "avg_words_per_chunk": [
            whole_chunks["chunk_text"]
            .str.split()
            .str.len()
            .mean(),

            fixed_chunks["chunk_text"]
            .str.split()
            .str.len()
            .mean(),

            section_chunks["chunk_text"]
            .str.split()
            .str.len()
            .mean(),
        ],
    }
)

display(chunk_summary)

,strategy,num_chunks,avg_words_per_chunk
0,Whole note,45,114.088889
1,Fixed words,56,97.571429
2,Section aware,148,34.689189


In [17]:
# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

In [ ]:
# ============================================================
# EMBEDDING MODEL SETUP
# ============================================================
#
# These are the SAME three embedding approaches used in the
# original 3 x 3 RAG configuration experiment.
#
# Gemini:
#   gemini-embedding-001
#
# BGE:
#   BAAI/bge-base-en-v1.5
#
# MedCPT:
#   Separate Query Encoder and Article Encoder
#
# IMPORTANT:
# - Document/chunk embeddings use the Article Encoder for MedCPT.
# - Retrieval queries use the Query Encoder for MedCPT.
# ============================================================




# -------------------------
# Gemini
# -------------------------

gemini_client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

GEMINI_EMBEDDING_MODEL = "gemini-embedding-001"


# -------------------------
# BGE
# -------------------------

BGE_MODEL_PATH = PROJECT_ROOT / "models" / "bge-base-en-v1.5"

bge_model = SentenceTransformer(
    str(BGE_MODEL_PATH)
)


# -------------------------
# MedCPT
# -------------------------

MEDCPT_QUERY_PATH = (
    PROJECT_ROOT
    / "models"
    / "MedCPT-Query-Encoder"
)

MEDCPT_ARTICLE_PATH = (
    PROJECT_ROOT
    / "models"
    / "MedCPT-Article-Encoder"
)

query_tokenizer = AutoTokenizer.from_pretrained(
    MEDCPT_QUERY_PATH,
    local_files_only=True
)

query_model = AutoModel.from_pretrained(
    MEDCPT_QUERY_PATH,
    local_files_only=True
)

article_tokenizer = AutoTokenizer.from_pretrained(
    MEDCPT_ARTICLE_PATH,
    local_files_only=True
)

article_model = AutoModel.from_pretrained(
    MEDCPT_ARTICLE_PATH,
    local_files_only=True
)

query_model.eval()
article_model.eval()

print("Gemini embedding client ready.")
print("BGE model loaded.")
print("MedCPT query/article models loaded.")